<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Compare simulation assumptions and compute annualized NPV comparison outputs.
- **Primary inputs**: runs/*/policies/indicator.csv
- **Primary outputs**: npv_annual.csv and comparison figures
- **Refactor role**: Keep in reporting layer; use shared run registry instead of hardcoded simulation folders.
- **Data policy**: keep existing simulation data in place during refactor; no run deletion in migration phases.


<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Map labels to run folders for model-assumption variants.
2. Load NPV-related indicators from each run.
3. Compare annual NPV metrics and export a consolidated table.

### Inputs
- `runs/<run_id>/policies/indicator.csv` for each run in `dict_simulation`

### Outputs
- `npv_annual.csv` in notebook output folder.
- Assumption-comparison plot/table views.

### How To Reuse
- Update `dict_simulation` entries to the run ids you want to compare.


In [ ]:
import pandas as pd
import os
from copy import deepcopy

from project.utils import get_json, get_pandas, get_series

In [ ]:
dict_simulation = {'No friction biased': 'policies_scenarios_20240520_121916',
                   'No friction unbiased': 'policies_scenarios_20240520_121945',
                   'Friction no present bias': 'policies_scenarios_20240520_163920',
                    'Friction without credit constraint': 'policies_scenarios_20240520_184553',
                   'Full friction': 'policies_scenarios_20240520_121930',
                   'Full friction hidden cost alternative': 'policies_scenarios_20240521_090215',
                  'Full friction thermal comfort': 'policies_scenarios_20240521_135902'
                   }
dict_simulation = {
    'No friction biased': 'policies_scenarios_20240521_185953',
    'No friction': 'policies_scenarios_20240521_185729',
                   'Full friction': 'policies_scenarios_20240521_190007',
    'Full friction biased': 'policies_scenarios_20240522_144821'
                   }

dict_simulation = {
    'Friction': 'policies_scenarios_20240527_120424',
    'Friction biased': 'policies_scenarios_20240527_120434',
    'No friction': 'policies_scenarios_20240527_120442',
    'No friction biased': 'policies_scenarios_20240527_120452'
}

folder_out = 'output_comparison'

In [ ]:
dict_output = {}
for k, f in dict_simulation.items():
  file = os.path.join('runs', f, 'policies/indicator.csv')
  dict_output.update({k: pd.read_csv(file, index_col=0)})
  
  
dict_output = pd.concat(dict_output, axis=0).rename_axis(['Scenario', 'Indicator'], axis=0)

dict_replace = {'carbon_tax' : 'Carbon tax',
                'carbon_tax_high' : 'Carbon tax + EU-ETS 2',
                'cee_2018' : 'WCO 2018',
                'cee_2021' : 'WCO 2021',
                'cee_2024' : 'WCO 2024',
                'obligation' : 'Mandatory renovation',
                'restriction_gas' : 'Ban gas',
                'subsidies_2018' : 'Subsidies 2018',
                'subsidies_2021' : 'Subsidies 2021',
                'subsidies_2024' : 'Subsidies 2024',
                'zero_interest_loan' : 'Zero interest loan'}

order = ['Carbon tax', 'Carbon tax + EU-ETS 2', 'WCO 2018', 'WCO 2021', 'WCO 2024', 'Subsidies 2018', 'Subsidies 2021', 'Subsidies 2024', 'Zero interest loan', 'Mandatory renovation', 'Ban gas']
dict_output = dict_output.rename(columns=dict_replace)
dict_output = dict_output.T
dict_output = dict_output.loc[[i for i in order if i in dict_output.index], :]


### NPV annual
Assessing the welfare implication under different modelling assumptions.

In [ ]:
temp = dict_output.xs('NPV annual (Billion euro/year)', level='Indicator', axis=1)
temp.to_csv(os.path.join(folder_out, 'npv_annual.csv'))
display(temp)

### Full friction

In [ ]:
dict_output